# Final cleanup

Final column standardisation and consolidation of the merged feature table.
See `log.md` for the record of manual decisions.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('df.tsv', sep='\t', encoding='utf-8')

In [ ]:
df.columns

In [ ]:
df_china = pd.read_excel('data/df_China.xlsx')

In [ ]:
common_cols = [col for col in df.columns if col in df_china.columns and col != 'isocodes']

# 2. Set 'isocode' as index in both dataframes temporarily for easy alignment
df_indexed = df.set_index('isocodes')
df_china_indexed = df_china.set_index('isocodes')

# 3. Update df_indexed with values from df_china_indexed for the common columns
df_indexed.update(df_china_indexed[common_cols])

# 4. Reset index to bring back 'isocode' as a column
df_updated = df_indexed.reset_index()

# 5. If you want to update your original df in place:
df.update(df_updated)

In [ ]:
wikidata_speakers = pd.read_json('data/wikidata_result.json')

In [ ]:
wikidata_speakers['num_of_speaker']

In [ ]:
wikidata_speakers_expanded = wikidata_speakers['num_of_speaker'].apply(pd.Series)

In [ ]:
wikidata_speakers_expanded

In [ ]:
wikidata_speakers = wikidata_speakers.drop(columns=['num_of_speaker']).join(wikidata_speakers_expanded)

In [ ]:
wikidata_speakers

In [ ]:
# Step 1: Explode the list of dictionaries so each dictionary becomes its own row
wikidata_speakers_exploded = wikidata_speakers.explode('num_of_speaker').reset_index(drop=True)

# Step 2: Expand the dictionaries in 'num_of_speakers' column into separate columns
num_speakers_expanded = wikidata_speakers_exploded['num_of_speaker'].apply(pd.Series)

# Step 3: Join the expanded columns back to the exploded dataframe (dropping original list column if you want)
df_final = wikidata_speakers_exploded.drop(columns=['num_of_speaker']).join(num_speakers_expanded)

In [ ]:
df_final

In [ ]:
df_final['number_of_speaker'] = df_final['number_of_speaker'].str.lstrip('+')

In [ ]:
df_final

In [ ]:
code_to_speakers = dict(zip(df_final['language_code'], df_final['number_of_speaker']))
name_to_speakers = dict(zip(df_final['language_name'], df_final['number_of_speaker']))

# Map 'isocodes' to number_of_speaker where possible
mapped_by_code = df['isocodes'].map(code_to_speakers)

# Map 'language_name' to number_of_speaker where possible
mapped_by_name = df['language_name'].map(name_to_speakers)

# Combine both mappings: prefer mapped_by_code, otherwise mapped_by_name
combined = mapped_by_code.combine_first(mapped_by_name)

# Update 'n_speakers' only where combined is not null
df['n_speakers'] = combined.where(combined.notna(), df['n_speakers'])

In [ ]:
df['n_speakers'] = df['n_speakers'].astype(str).str.replace(r'[+,]', '', regex=True)

# Then convert to integers, forcing errors to NaN
df['n_speakers'] = pd.to_numeric(df['n_speakers'], errors='coerce').fillna(0).astype(int)

In [ ]:
df.sort_values(by='n_speakers', ascending = False)

In [ ]:
df['root_node'] = df['root_node'].replace(r'.*\[[a-z0-9]+\]\[[a-z0-9]{3}\]-l-', 'Isolate', regex=True)
df['root_node'] = df['root_node'].fillna('Potentially isolate')

In [ ]:
df = df.drop(columns=['phones'])
df = df.drop(columns=['Unnamed: 0'])

In [ ]:
df

In [ ]:
df.columns

In [ ]:
fancy_names = ['Glottocode', 'Language Name', 'ISO code', 'Language or Dialect?', 'Macroarea', 'Latitude', 'Longitude', 'Existence of Dictionary', 'Language Family', 'Existence of a Bible',
             'Gigabytes of Information in Common Crawl','Gigabytes of Information in Archives of Linguistic Documentation', 'Number of Datasets in HuggingFace', 'Number of Articles on Wikipedia', 
              'Gigabytes of Information in Opus', 'Lowercase Name', 'Gigabytes of Information in XEUS', 'Gigabytes of Information in Wikitongues', 'Number of Models in HuggingFace', 
               'Mean Number of Downloads in HuggingFace Models', 'Earliest Date of Upload in HuggingFace Models', 'Number of Active Users in Wikipedia', 'Institutional Status', 
               'Level of Endangerment', 'Main Country where Spoken', 'Code of Administrative Unit 1 (~Region) where Spoken', 'ISO of Main Country where Spoken', 'ISO of Other Countries where Spoken', 
               'Spoken in Contested Area?' 'Number of Speakers', 'Geometry', 'Closest University to the Language', 'Second Closest University to the Language', 'Distance to Closest University (in Km)', 'Distance to Second Closest University (in Km)', 
                'Human Development Index (Aministrative Unit 1 Level)', 'Gross Domestic Product (Administrative Unit 1 Level), Existence of Cybersecurity Legislation/Regulation', 
                   'Percentage of Household with Phones (National Level)', 'Percentage of Households with Home Internet (National Level)', 'Percentage of Individuals Using the Internet (National Level)', 
                      'Average Download Speed (in Kbps - approximately Km² Level)', 'Average Upload Speed (in Kpbs - approximately Km² Level)' , 'Average Latency (in Ms - approximately Km² Level)', 
                      'Percentage of Graduates in Science, Technology, Engineering, and Mathematics (STEM)',
                         'Literacy Rate (National Level)', 'Educational Attainment Level (Aministrative Unit 1 Level)', 'Research and Development Spending as a Percentage of Gross Domestic Product']

In [ ]:
software_friendly_names = [
    'glottocode', 'language_name', 'iso_code', 'language_or_dialect', 'macroarea', 'latitude', 'longitude',
    'dictionary_exists', 'language_family', 'bible_exists',
    'cc_gb', 'archival_gb', 'datasets_n', 'wiki_articles_n',
    'opus_gb', 'lowercase_name', 'xeus_gb', 'wikitongues_gb', 'models_n',
    'models_mean_downloads', 'models_earliest_date', 'wiki_active_users', 'institutional_status',
    'endangerment_status', 'primary_country', 'admin_1_code', 'country_iso_code', 'other_country_iso_codes',
    'contested_area', 'n_speakers', 'geometry', 'closest_university', 'second_closest_university',
    'distance_to_closest_university_km', 'distance_to_second_closest_university_km',
    'admin_1_hdi', 'admin_1_gdp', 'cybersecurity_law_exists',
    'households_with_phones_pct', 'households_with_home_internet_pct', 'individuals_using_internet_pct',
    'avg_download_speed_kbps', 'avg_upload_speed_kbps', 'avg_latency_ms','pct_stem_grads',
    'literacy_rate', 'admin_1_education_level', 'rnd_gdp_pct'
]

In [ ]:
len(software_friendly_names)

In [ ]:
len(df.columns)

In [ ]:
fancy_to_friendly_map = {
    'Glottocode': 'glottocode',
    'Language Name': 'language_name',
    'ISO code': 'iso_code',
    'Language or Dialect?': 'language_or_dialect',
    'Macroarea': 'macroarea',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'Existence of Dictionary': 'has_dictionary',
    'Language Family': 'language_family',
    'Existence of a Bible': 'has_bible',
    'Gigabytes of Information in Common Crawl': 'gb_common_crawl',
    'Gigabytes of Information in Archives of Linguistic Documentation': 'gb_archives',
    'Number of Datasets in HuggingFace': 'n_datasets_hf',
    'Number of Articles on Wikipedia': 'n_wikipedia_articles',
    'Gigabytes of Information in Opus': 'gb_opus',
    'Lowercase Name': 'lowercase_name',
    'Gigabytes of Information in XEUS': 'gb_xeus',
    'Gigabytes of Information in Wikitongues': 'gb_wikitongues',
    'Number of Models in HuggingFace': 'n_models_hf',
    'Mean Number of Downloads in HuggingFace Models': 'mean_model_downloads_hf',
    'Earliest Date of Upload in HuggingFace Models': 'earliest_model_upload_hf',
    'Number of Active Users in Wikipedia': 'n_wikipedia_users',
    'Institutional Status': 'institutional_status',
    'Level of Endangerment': 'endangerment_status',
    'Main Country where Spoken': 'primary_country',
    'Code of Administrative Unit 1 (~Region) where Spoken': 'admin_1_code',
    'ISO of Main Country where Spoken': 'country_iso_code',
    'ISO of Other Countries where Spoken': 'other_country_iso_codes',
    'Spoken in Contested Area?Number of Speakers': 'contested_area_speakers',
    'Geometry': 'geometry',
    'Closest University to the Language': 'closest_university',
    'Second Closest University to the Language': 'second_closest_university',
    'Distance to Closest University (in Km)': 'distance_to_closest_uni_km',
    'Distance to Second Closest University (in Km)': 'distance_to_second_closest_uni_km',
    'Human Development Index (Aministrative Unit 1 Level)': 'hdi_admin_1',
    'Gross Domestic Product (Administrative Unit 1 Level), Existence of Cybersecurity Legislation/Regulation': 'gdp_admin_1_cyber_law',
    'Percentage of Household with Phones (National Level)': 'pct_phones',
    'Percentage of Households with Home Internet (National Level)': 'pct_home_internet',
    'Percentage of Individuals Using the Internet (National Level)': 'pct_individuals_internet',
    'Average Download Speed (in Kbps - approximately Km² Level)': 'avg_download_kbps',
    'Average Upload Speed (in Kpbs - approximately Km² Level)': 'avg_upload_kbps',
    'Average Latency (in Ms - approximately Km² Level)': 'avg_latency_ms',
    'Percentage of Graduates in Science, Technology, Engineering, and Mathematics (STEM)': 'pct_stem_grads',
    'Literacy Rate (National Level)': 'literacy_rate',
    'Educational Attainment Level (Aministrative Unit 1 Level)': 'education_attainment_admin_1',
    'Research and Development Spending as a Percentage of Gross Domestic Product': 'rnd_pct_gdp'
}

In [ ]:
df.columns = software_friendly_names

In [ ]:
df.columns

In [ ]:
df['contested_area'] = df['contested_area'].apply(lambda x: 1 if x == 1 else 0)

In [ ]:
df

In [ ]:
df.loc[df['language_name'] == 'Standard Indonesian', 'n_speakers'] = 252400000
df.loc[df['language_name'] == 'English', 'n_speakers'] = 1350000000

In [ ]:
df.to_csv('final_df.csv')